In [19]:
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
import numpy as np
# 参考: https://blog.csdn.net/qq_40042726/article/details/121192531

In [10]:
# 实例化
writer = SummaryWriter('./log') # 会自动创建文件

<span style='color:red;'></span>

<span style='color:red;'>1. 可视化标量数据</span>

In [11]:
for i in range(100):
    loss = i
    writer.add_scalar('loss',loss,i) # 名称, 要记录的标量(相当于y值), x值
# writer.close()

<span style='color:red;'>2.可视化网络结构<span>

In [14]:
import torch
import torch.nn as nn

# 实例化
writer = SummaryWriter('./log') # 会自动创建文件
class Model(nn.Module):
    def __init__(self):
        super(Model,self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1,64,kernel_size=3,stride=1,padding=1),
            nn.ReLU(),
            nn.Conv2d(64,128,kernel_size=3,stride=1,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(stride=2,kernel_size=2)
        )
        self.dense = nn.Sequential(
            nn.Linear(14*14*128,1024),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(1024,10)
        )
    def forward(self,x):
        x = self.conv1(x)
        print('x.shape:',x.shape)
        x = x.view(-1,14*14*128)
        x = self.dense(x)
        return x

model = Model()
images = torch.randn(1,1,28,28)
writer.add_graph(model,images)
# writer.close()

x.shape: torch.Size([1, 128, 14, 14])


f:\my_softers\project_IDE\Anconda\envs\jp_layout_pytorch\lib\site-packages\ipykernel_launcher.py:21: TracerWarning: Converting a tensor to a Python integer might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!


x.shape: torch.Size([1, 128, 14, 14])
x.shape: torch.Size([1, 128, 14, 14])


<span style='color:red;'>3.可视化图片<span>

In [20]:
image_pil = Image.open('./imgs/1.jpg')
img_np = np.array(image_pil)
writer.add_image('test', # 保存图片的名称
                 img_np, # 图片的类型要是 torch.tensor, numpy, string中的一种
                 1,      # 第几张图片
                 dataformats='HWC') # 默认是CHW, tensor是CHW, numpy是HWC
# writer.close()

<span style='color:red;'>4.可视化卷积层<span>

In [15]:
import torch
import cv2
import torch.nn as nn
import torchvision
from torchvision import transforms
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import os
import cv2
from PIL import Image
import numpy as np
# 实例化
writer = SummaryWriter('./log') # 会自动创建文件

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.convl = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(stride=2, kernel_size=2)
        )

        self.dense = torch.nn.Sequential(
            torch.nn.Linear(14 * 14 * 128, 1024),
            torch.nn.ReLU(),
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(1024, 10)
        )

    def forward(self, x):
        x = self.convl(x)

        x = x.view(-1, 14 * 14 * 128) 
        x = self.dense(x)
        output = F.log_softmax(x, dim=1)
        return output

pipline = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize((0.1307,),(0.3081))
])


def hook_func(module, input):
    x = input[0][0]
    x = x.unsqueeze(1)
    global i
    image_batch = torchvision.utils.make_grid(x, padding=4)
    image_batch = image_batch.numpy().transpose(1, 2, 0)
    writer.add_image("test", image_batch, i, dataformats='HWC')
    i += 1

model = MyModel()
i = 0
for name, m in model.named_modules():
    m.register_forward_pre_hook(hook_func)
img = Image.open('./imgs/1.jpg')
writer.add_image('img',np.array(img),1,dataformats='HWC')
print('='*6)
# print(img.shape)
# img = pipline(img).unsqueeze(0)
img = transforms.functional.resize(img,[28,28])
print('img.shape:',np.array(img).shape)
img = np.array(img)[...,0].reshape(28,28,1)
with torch.no_grad():
    print('img1.shape:',np.array(img).shape)
    input = pipline(img)
    input = input[None,...]
    model(input)

img.shape: (28, 28, 3)
img1.shape: (28, 28, 1)


<span style='color:red;'> *显示出tensorboard记录的数据</span>  
在终端输入 tensorboard --logdir=logs # logs为tensorboard记录